## B. Statistics & Hypothesis Testing (Monthly)

This section uses `combined_goldandnflx_monthly.csv` (monthly merged data) to produce:

- Descriptive statistics (mean, standard deviation, etc.)
- Correlation analysis (`df.corr()`)
- Linear regression model (`statsmodels.OLS`)
- Statistical significance of regression coefficients: t-tests and p-values (with clear conclusions)


In [ ]:
import pandas as pd
import numpy as np

import statsmodels.api as sm
from scipy import stats

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)


In [ ]:
# Load the monthly merged dataset
path = "combined_goldandnflx_monthly.csv"
df = pd.read_csv(path)

# Create a Date column for sorting / sanity checks (consistent with the visualization notebook)
df["Date"] = pd.to_datetime(df["Year"].astype(str) + "-" + df["Month"].astype(str).str.zfill(2) + "-01")
df = df.sort_values("Date").reset_index(drop=True)

# Numeric columns used in this section
num_cols = ["Gold_USD_m_mean", "NFLX_AdjClose_m_mean"]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")

print(df.head())
print("\nShape:", df.shape)
print("\nMissing values:\n", df[num_cols].isna().sum())


### 1) Descriptive statistics (mean, standard deviation, etc.)

We compute basic descriptive statistics for monthly gold prices and monthly Netflix prices, and then extract **mean** and **standard deviation** as numbers you can directly cite in the report.


In [ ]:
desc = df[num_cols].describe().T
# Optionally add variance / coefficient of variation

desc["var"] = df[num_cols].var(numeric_only=True)
desc["cv"] = desc["std"] / desc["mean"]

display(desc)

summary_mean_std = desc[["mean", "std"]].rename(columns={"mean": "Mean", "std": "Std"})
display(summary_mean_std)


### 2) Correlation analysis (Pearson)

We compute the Pearson correlation using `df.corr()` and also report the significance test (p-value), so you can state whether the correlation is statistically significant.


In [ ]:
corr = df[num_cols].corr()  # df.corr()
display(corr)

# Correlation significance test (H0: rho = 0)
sub = df[num_cols].dropna()
r, p = stats.pearsonr(sub["Gold_USD_m_mean"], sub["NFLX_AdjClose_m_mean"])

print(f"Pearson r = {r:.4f}")
print(f"p-value   = {p:.4g}")


### 3) Linear regression (OLS)

Model setup (ready to paste into your report):

- Dependent variable (Y): monthly Netflix price `NFLX_AdjClose_m_mean`
- Independent variable (X): monthly average gold price `Gold_USD_m_mean`

We run a standard OLS regression and then test coefficient significance using t-tests:

- **H0**: coefficient = 0 (not significant)
- **H1**: coefficient ≠ 0 (significant)


In [ ]:
# Regression: NFLX ~ Gold
reg_df = df[["NFLX_AdjClose_m_mean", "Gold_USD_m_mean"]].dropna().copy()

X = sm.add_constant(reg_df[["Gold_USD_m_mean"]])
y = reg_df["NFLX_AdjClose_m_mean"]

model = sm.OLS(y, X).fit()
print(model.summary())


In [ ]:
# Coefficient t-tests (same as in summary, formatted as a table for easy reporting)
coef_table = pd.DataFrame({
    "coef": model.params,
    "std_err": model.bse,
    "t": model.tvalues,
    "p_value": model.pvalues,
    "ci_low": model.conf_int()[0],
    "ci_high": model.conf_int()[1],
})

display(coef_table)

alpha = 0.05
sig = coef_table["p_value"] < alpha
print("\nSignificant at alpha=0.05?\n", sig)


### 4) Report-ready conclusion text (auto-generated)

This cell automatically generates short English statements based on your results (correlation, regression slope, and p-values), reducing manual copy/paste errors.


In [ ]:
# Auto-generate short report-ready statements
slope = model.params["Gold_USD_m_mean"]
slope_p = model.pvalues["Gold_USD_m_mean"]
intercept = model.params["const"]
r2 = model.rsquared

corr_r, corr_p = r, p

corr_sig_text = "statistically significant" if corr_p < 0.05 else "not statistically significant"
reg_sig_text = "statistically significant" if slope_p < 0.05 else "not statistically significant"

print(
    "Correlation: In the monthly data, the Pearson correlation between gold price and Netflix price is "
    f"r = {corr_r:.3f} (p = {corr_p:.3g}); at the 5% level, the correlation is {corr_sig_text}."
)

print(
    "Regression: The OLS regression (NFLX ~ Gold) estimates a gold-price slope of "
    f"{slope:.4f} (p = {slope_p:.3g}); at the 5% level, this coefficient is {reg_sig_text}. "
    f"Model R^2 = {r2:.3f}."
)

print(
    "Model form: NFLX_AdjClose_m_mean = "
    f"{intercept:.4f} + {slope:.4f} * Gold_USD_m_mean + ε"
)
